<a href="https://colab.research.google.com/github/indahsarikurniawati/tugas-ml/blob/main/tugas_3_klasifikasi_ml_23_april_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AIR QUALITY DATA IN INDIA (2015-2020)

Kualitas udara merupakan salah satu faktor penting yang memengaruhi kesehatan manusia dan lingkungan. Peningkatan aktivitas industri, transportasi, serta urbanisasi menyebabkan konsentrasi polutan udara seperti PM2.5, PM10, karbon monoksida (CO), nitrogen dioksida (NO₂), dan ozon (O₃) semakin meningkat. Kondisi ini mendorong perlunya analisis yang mampu mengidentifikasi dan mengklasifikasikan kualitas udara secara akurat.

Penelitian ini menggunakan dataset kualitas udara yang diperoleh dari platform Kaggle, khususnya data agregat harian tingkat kota (city_day). Dataset tersebut memuat berbagai parameter polutan serta nilai Air Quality Index (AQI) yang digunakan sebagai indikator utama tingkat pencemaran udara. Untuk keperluan analisis klasifikasi, nilai AQI kemudian dikategorikan menjadi beberapa kelas, yaitu baik, sedang, dan buruk.

Metode yang digunakan dalam penelitian ini adalah metode klasifikasi dengan pendekatan machine learning. Beberapa algoritma yang digunakan antara lain Random Forest sebagai metode bagging, XGBoost sebagai metode boosting, serta Artificial Neural Network (ANN). Ketiga model tersebut dibandingkan berdasarkan performa prediksi menggunakan metrik evaluasi seperti akurasi, precision, recall, dan F1-score. Hasil dari penelitian ini diharapkan dapat memberikan gambaran mengenai model terbaik dalam mengklasifikasikan kualitas udara serta membantu dalam pengambilan keputusan terkait pengelolaan lingkungan.


In [ ]:
import pandas as pd
import numpy as np

# visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# model
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

# evaluasi
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
data = pd.read_csv("/content/city_day.csv")
data.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [ ]:
data.info()
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB


,0
City,0
Date,0
PM2.5,4598
PM10,11140
NO,3582
NO2,3585
NOx,4185
NH3,10328
CO,2059
SO2,3854


Berdasarkan hasil pengecekan struktur data, dataset kualitas udara yang digunakan memiliki jumlah observasi sebanyak 29.531 baris dengan 16 variabel. Dataset ini tergolong cukup besar sehingga dinilai memadai untuk digunakan dalam analisis machine learning, khususnya klasifikasi. Variabel yang tersedia didominasi oleh data numerik yang merepresentasikan konsentrasi berbagai polutan udara seperti PM2.5, PM10, NO₂, CO, dan O₃, serta terdapat beberapa variabel kategorik seperti nama kota dan tanggal pengamatan.

Namun demikian, ditemukan bahwa dataset memiliki sejumlah nilai yang hilang (missing values) pada beberapa variabel. Beberapa variabel bahkan memiliki jumlah data kosong yang cukup tinggi, seperti PM10, NH₃, dan Xylene. Kondisi ini menunjukkan bahwa data kualitas udara yang digunakan merupakan data riil yang tidak sepenuhnya lengkap, sehingga memerlukan tahap pembersihan data sebelum dilakukan analisis lebih lanjut.


In [ ]:
# hapus kolom yang tidak dipakai
data = data.drop(['City', 'Date', 'Xylene'], axis=1)

In [ ]:
data = data.dropna(subset=['AQI'])

In [ ]:
data.isnull().sum()

,0
PM2.5,678
PM10,7086
NO,387
NO2,391
NOx,1857
NH3,6536
CO,445
SO2,605
O3,807
Benzene,3535


Berdasarkan hasil pengecekan ulang terhadap missing value setelah dilakukan penghapusan beberapa variabel dan data tanpa nilai AQI, masih terdapat sejumlah nilai yang hilang pada beberapa variabel numerik. Variabel seperti PM10, NH3, dan Toluene masih memiliki jumlah data kosong yang cukup besar dibandingkan variabel lainnya.

Hal ini menunjukkan bahwa meskipun sebagian data telah dibersihkan, dataset kualitas udara masih memiliki ketidaksempurnaan yang umum terjadi pada data riil. Oleh karena itu, diperlukan langkah lanjutan untuk menangani missing value agar tidak memengaruhi kinerja model dalam proses klasifikasi.

Dalam penelitian ini, penanganan missing value dilakukan menggunakan metode imputasi median pada setiap variabel numerik. Metode ini dipilih karena lebih robust terhadap outlier dan mampu mempertahankan distribusi data dibandingkan metode rata-rata. Setelah proses imputasi dilakukan, diharapkan tidak ada lagi nilai kosong sehingga dataset siap digunakan dalam tahap pemodelan.

In [ ]:
data = data.drop(['NH3', 'Toluene'], axis=1)

In [ ]:
for col in data.columns:
    if data[col].dtype != 'object':
        data[col] = data[col].fillna(data[col].median())

In [ ]:
data.isnull().sum()

,0
PM2.5,0
PM10,0
NO,0
NO2,0
NOx,0
CO,0
SO2,0
O3,0
Benzene,0
AQI,0


Setelah dilakukan proses pembersihan data, seluruh variabel yang digunakan dalam penelitian ini sudah tidak memiliki nilai yang hilang. Hal ini menunjukkan bahwa proses preprocessing yang dilakukan, baik melalui penghapusan variabel dengan tingkat missing value tinggi maupun imputasi menggunakan median, telah berhasil menghasilkan dataset yang lengkap dan siap digunakan.

In [ ]:
# memastikan label sudah ada
def kategori_aqi(aqi):
    if aqi <= 50:
        return 'Baik'
    elif aqi <= 100:
        return 'Sedang'
    else:
        return 'Buruk'

data['Kategori'] = data['AQI'].apply(kategori_aqi)

In [ ]:
# encoding target
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Kategori'] = le.fit_transform(data['Kategori'])

In [ ]:
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Baik': np.int64(0), 'Buruk': np.int64(1), 'Sedang': np.int64(2)}


Hasil proses encoding menunjukkan bahwa variabel kategori kualitas udara telah berhasil dikonversi ke dalam bentuk numerik. Kategori “Baik” direpresentasikan dengan nilai 0, “Buruk” dengan nilai 1, dan “Sedang” dengan nilai 2.

Proses ini dilakukan agar data dapat diproses oleh algoritma machine learning yang umumnya hanya menerima input dalam bentuk numerik. Meskipun nilai numerik diberikan pada setiap kategori, tidak terdapat makna urutan (ordinal) di antara kategori tersebut, melainkan hanya sebagai representasi label.

Dengan demikian, variabel target telah siap digunakan dalam proses pemodelan klasifikasi untuk memprediksi kategori kualitas udara berdasarkan nilai polutan yang tersedia.


In [ ]:
# pemilihan fitur
X = data[['PM2.5', 'PM10', 'NO2', 'CO', 'O3']]
y = data['Kategori']

Pada tahap ini dilakukan pemilihan fitur yang akan digunakan dalam proses pemodelan. Variabel yang dipilih merupakan parameter utama kualitas udara, yaitu PM2.5, PM10, NO2, CO, dan O3. Variabel-variabel ini dipilih karena memiliki kontribusi signifikan terhadap nilai AQI. Sementara itu, variabel Kategori digunakan sebagai target dalam proses klasifikasi.


In [ ]:
# split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

Dataset dibagi menjadi data training dan data testing dengan proporsi 80:20. Data training digunakan untuk melatih model, sedangkan data testing digunakan untuk menguji performa model terhadap data yang belum pernah dilihat sebelumnya.


In [ ]:
# model 1 : random forest (bagging)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc_rf = accuracy_score(y_test, pred_rf)
print("Accuracy RF:", acc_rf)

print(classification_report(y_test, pred_rf))

Accuracy RF: 0.8857142857142857
              precision    recall  f1-score   support

           0       0.79      0.61      0.69       282
           1       0.93      0.94      0.93      3033
           2       0.82      0.84      0.83      1655

    accuracy                           0.89      4970
   macro avg       0.85      0.80      0.82      4970
weighted avg       0.88      0.89      0.88      4970



Berdasarkan hasil pengujian menggunakan model Random Forest, diperoleh nilai akurasi sebesar 0,8857 atau sekitar 88,57%. Hal ini menunjukkan bahwa model mampu mengklasifikasikan kualitas udara dengan tingkat ketepatan yang cukup tinggi.

Jika dilihat lebih rinci berdasarkan masing-masing kelas, model menunjukkan performa yang berbeda. Pada kategori “Buruk”, model memiliki nilai precision dan recall yang sangat tinggi, yaitu masing-masing sebesar 0,93 dan 0,94. Hal ini menunjukkan bahwa model sangat baik dalam mengenali kondisi kualitas udara yang buruk.

Sementara itu, pada kategori “Sedang”, model juga menunjukkan performa yang cukup baik dengan nilai f1-score sebesar 0,83. Hal ini mengindikasikan bahwa model cukup seimbang dalam mengklasifikasikan data pada kategori tersebut.

Namun, pada kategori “Baik”, performa model masih relatif lebih rendah dibandingkan kategori lainnya, dengan nilai recall sebesar 0,61 dan f1-score sebesar 0,69. Hal ini menunjukkan bahwa model masih kesulitan dalam mengenali secara tepat kondisi kualitas udara yang tergolong baik.

Selain itu, berdasarkan nilai support terlihat bahwa distribusi data tidak seimbang, di mana kategori “Buruk” memiliki jumlah data yang jauh lebih banyak dibandingkan kategori lainnya. Ketidakseimbangan ini dapat memengaruhi performa model, sehingga model cenderung lebih baik dalam memprediksi kelas mayoritas.

Secara keseluruhan, model Random Forest sudah menunjukkan performa yang baik dalam mengklasifikasikan kualitas udara, terutama dalam mendeteksi kondisi udara yang buruk, meskipun masih terdapat ruang perbaikan dalam mengenali kategori udara yang baik.


In [ ]:
# model 2 : XGBoost
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
# training model XGBOOST
xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [15:52:39] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
# evaluasi model XGBOOST
from sklearn.metrics import accuracy_score, classification_report

acc_xgb = accuracy_score(y_test, pred_xgb)
print("Accuracy XGBoost:", acc_xgb)

print(classification_report(y_test, pred_xgb))

Accuracy XGBoost: 0.8830985915492958
              precision    recall  f1-score   support

           0       0.76      0.62      0.68       282
           1       0.93      0.93      0.93      3033
           2       0.82      0.84      0.83      1655

    accuracy                           0.88      4970
   macro avg       0.83      0.80      0.81      4970
weighted avg       0.88      0.88      0.88      4970



Berdasarkan hasil pengujian menggunakan model XGBoost, diperoleh nilai akurasi sebesar 0,8831 atau sekitar 88,31%. Hasil ini menunjukkan bahwa model memiliki kemampuan yang baik dalam mengklasifikasikan kualitas udara berdasarkan variabel polutan yang digunakan.

Jika dilihat dari masing-masing kategori, model menunjukkan performa yang sangat baik pada kategori “Buruk”, dengan nilai precision dan recall sebesar 0,93. Hal ini menunjukkan bahwa model mampu mengenali kondisi kualitas udara yang buruk dengan sangat akurat.

Pada kategori “Sedang”, model juga menunjukkan performa yang cukup baik dengan nilai f1-score sebesar 0,83, yang mengindikasikan keseimbangan antara precision dan recall dalam mengklasifikasikan kategori tersebut.

Namun, pada kategori “Baik”, performa model masih relatif lebih rendah dibandingkan kategori lainnya, dengan nilai recall sebesar 0,62 dan f1-score sebesar 0,68. Hal ini menunjukkan bahwa model masih mengalami kesulitan dalam mengidentifikasi kondisi kualitas udara yang tergolong baik.

Secara keseluruhan, performa model XGBoost tidak berbeda jauh dengan Random Forest, dengan tingkat akurasi yang hampir sama. Hal ini menunjukkan bahwa kedua model memiliki kemampuan yang sebanding dalam menangkap pola pada data kualitas udara.


In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(y_test, pred_xgb)
print(cm_xgb)

[[ 176    3  103]
 [   1 2820  212]
 [  55  207 1393]]


Berdasarkan hasil confusion matrix, dapat dilihat bagaimana model XGBoost mengklasifikasikan setiap kategori kualitas udara. Pada kategori “Baik”, dari total 282 data, model berhasil mengklasifikasikan dengan benar sebanyak 176 data, sementara sisanya salah diklasifikasikan sebagai kategori “Sedang” dan “Buruk”. Hal ini menunjukkan bahwa model masih mengalami kesulitan dalam mengenali kategori kualitas udara yang baik secara konsisten.

Pada kategori “Buruk”, model menunjukkan performa yang sangat baik. Dari total 3033 data, sebanyak 2820 data berhasil diklasifikasikan dengan benar, dan hanya sebagian kecil yang salah diklasifikasikan ke kategori lain. Hal ini menunjukkan bahwa model sangat mampu mendeteksi kondisi kualitas udara yang buruk.

Sementara itu, pada kategori “Sedang”, dari total 1655 data, sebanyak 1393 data berhasil diklasifikasikan dengan benar. Namun masih terdapat sejumlah data yang salah diklasifikasikan, baik ke kategori “Baik” maupun “Buruk”. Hal ini menunjukkan bahwa kategori “Sedang” masih memiliki tingkat ambiguitas yang cukup tinggi sehingga lebih sulit diprediksi secara tepat.

Secara keseluruhan, confusion matrix menunjukkan bahwa model memiliki performa terbaik dalam mengklasifikasikan kategori “Buruk”, diikuti oleh kategori “Sedang”, dan paling rendah pada kategori “Baik”. Hal ini sejalan dengan distribusi data yang tidak seimbang, di mana kategori “Buruk” memiliki jumlah data paling banyak sehingga model lebih terlatih untuk mengenali pola pada kategori tersebut.


In [ ]:
# scalling data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# train model 3 : neural network
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=500,
    random_state=42
)

nn.fit(X_train_scaled, y_train)

pred_nn = nn.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

acc_nn = accuracy_score(y_test, pred_nn)
print("Accuracy NN:", acc_nn)

print(classification_report(y_test, pred_nn))

Accuracy NN: 0.8792756539235412
              precision    recall  f1-score   support

           0       0.70      0.56      0.63       282
           1       0.94      0.93      0.93      3033
           2       0.80      0.85      0.83      1655

    accuracy                           0.88      4970
   macro avg       0.81      0.78      0.79      4970
weighted avg       0.88      0.88      0.88      4970



Berdasarkan hasil pengujian menggunakan model Artificial Neural Network (ANN), diperoleh nilai akurasi sebesar 0,8793 atau sekitar 87,93%. Hasil ini menunjukkan bahwa model memiliki kemampuan yang cukup baik dalam mengklasifikasikan kualitas udara, meskipun sedikit lebih rendah dibandingkan model lainnya.

Jika dilihat berdasarkan masing-masing kategori, model menunjukkan performa yang sangat baik pada kategori “Buruk”, dengan nilai precision sebesar 0,94 dan recall sebesar 0,93. Hal ini menunjukkan bahwa model mampu mengenali kondisi kualitas udara yang buruk dengan sangat baik.

Pada kategori “Sedang”, model juga memiliki performa yang cukup baik dengan nilai f1-score sebesar 0,83, yang menunjukkan keseimbangan antara precision dan recall.

Namun, pada kategori “Baik”, performa model relatif lebih rendah dengan nilai recall sebesar 0,56 dan f1-score sebesar 0,63. Hal ini menunjukkan bahwa model masih mengalami kesulitan dalam mengidentifikasi kategori kualitas udara yang baik.

Secara keseluruhan, model Neural Network menunjukkan performa yang baik, namun masih berada sedikit di bawah model Random Forest dan XGBoost dalam hal akurasi.


In [ ]:
# confusion matrix
from sklearn.metrics import confusion_matrix

cm_nn = confusion_matrix(y_test, pred_nn)
print(cm_nn)

[[ 159    3  120]
 [   5 2807  221]
 [  62  189 1404]]


Berdasarkan hasil confusion matrix pada model Artificial Neural Network, dapat dilihat bagaimana model mengklasifikasikan setiap kategori kualitas udara. Pada kategori “Baik”, dari total 282 data, model berhasil mengklasifikasikan dengan benar sebanyak 159 data, sementara sisanya salah diklasifikasikan ke dalam kategori “Sedang” dan “Buruk”. Hal ini menunjukkan bahwa model masih mengalami kesulitan dalam mengenali kategori kualitas udara yang baik secara akurat.

Pada kategori “Buruk”, model menunjukkan performa yang sangat baik. Dari total 3033 data, sebanyak 2807 data berhasil diklasifikasikan dengan benar, dan hanya sebagian kecil yang salah diklasifikasikan ke kategori lain. Hal ini menunjukkan bahwa model sangat efektif dalam mendeteksi kondisi kualitas udara yang buruk.

Sementara itu, pada kategori “Sedang”, dari total 1655 data, sebanyak 1404 data berhasil diklasifikasikan dengan benar. Namun masih terdapat beberapa kesalahan klasifikasi ke kategori “Baik” dan “Buruk”, yang menunjukkan bahwa kategori ini memiliki tingkat kesulitan yang lebih tinggi untuk diprediksi secara tepat.

Secara keseluruhan, hasil confusion matrix menunjukkan bahwa model Neural Network memiliki performa terbaik pada kategori “Buruk”, diikuti oleh kategori “Sedang”, dan paling rendah pada kategori “Baik”. Pola ini konsisten dengan model sebelumnya dan dipengaruhi oleh distribusi data yang tidak seimbang.


In [ ]:
import pandas as pd

hasil = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Neural Network'],
    'Accuracy': [acc_rf, acc_xgb, acc_nn]
})

print(hasil)

            Model  Accuracy
0   Random Forest  0.885714
1         XGBoost  0.883099
2  Neural Network  0.879276


Berdasarkan hasil perbandingan performa model, diperoleh bahwa model Random Forest memiliki nilai akurasi tertinggi sebesar 0,8857, diikuti oleh XGBoost sebesar 0,8831, dan Neural Network sebesar 0,8793.

Perbedaan nilai akurasi antar model relatif kecil, yang menunjukkan bahwa ketiga model memiliki kemampuan yang hampir setara dalam mengklasifikasikan kualitas udara. Namun demikian, Random Forest tetap menjadi model terbaik karena memiliki nilai akurasi paling tinggi serta performa yang lebih stabil dibandingkan model lainnya.

XGBoost sebagai metode boosting menunjukkan performa yang kompetitif dan mendekati Random Forest, sementara Neural Network memiliki performa yang sedikit lebih rendah. Hal ini dapat disebabkan oleh karakteristik data yang lebih cocok ditangani oleh model berbasis tree seperti Random Forest dan XGBoost.

Dengan demikian, dapat disimpulkan bahwa model Random Forest merupakan model yang paling optimal dalam penelitian ini untuk mengklasifikasikan kualitas udara berdasarkan variabel polutan yang digunakan.


Berdasarkan distribusi data, diketahui bahwa jumlah observasi pada masing-masing kategori kualitas udara tidak seimbang. Kategori “Buruk” memiliki jumlah data yang jauh lebih banyak dibandingkan kategori “Sedang” dan “Baik”. Kondisi ini menunjukkan bahwa dataset yang digunakan termasuk dalam kategori imbalanced dataset atau data tidak seimbang.

Ketidakseimbangan ini dapat disebabkan oleh karakteristik data yang merupakan data riil (real-world data). Dalam kondisi nyata, kualitas udara di wilayah perkotaan, khususnya di negara berkembang dengan tingkat industrialisasi dan mobilitas tinggi, cenderung lebih sering berada pada kondisi yang kurang baik. Hal ini menyebabkan nilai AQI lebih banyak berada pada kategori “Buruk”.

Selain itu, faktor seperti kepadatan penduduk, aktivitas kendaraan bermotor, emisi industri, serta kondisi lingkungan tertentu seperti musim kemarau dan pembakaran terbuka turut berkontribusi terhadap tingginya tingkat pencemaran udara. Akibatnya, distribusi data menjadi tidak merata dan didominasi oleh kategori tertentu.

Kondisi data yang tidak seimbang ini berdampak pada performa model machine learning, di mana model cenderung lebih baik dalam mengklasifikasikan kategori mayoritas dibandingkan kategori minoritas. Hal ini terlihat dari hasil evaluasi model yang menunjukkan performa tinggi pada kategori “Buruk” namun lebih rendah pada kategori “Baik”.

Meskipun demikian, penggunaan dataset yang tidak seimbang tetap dapat diterima dalam penelitian ini karena mencerminkan kondisi nyata di lapangan. Oleh karena itu, hasil analisis yang diperoleh tetap valid, dengan catatan adanya keterbatasan terkait distribusi data yang tidak merata.
